# 02 - Transform WLASL to Universal CSV

Transforms the WLASL dataset into the shared universal CSV format.
Output is saved as a single WLASL-specific CSV in `kaggle_dataset/` (universal schema).

In [10]:
from pathlib import Path
import json
import cv2
import pandas as pd

# Path configuration
PROJECT_ROOT = Path.cwd().parents[1]
WLASL_JSON = PROJECT_ROOT / 'kaggle_dataset' / 'WLASL_v0.3.json'
VIDEOS_DIR = PROJECT_ROOT / 'kaggle_dataset' / 'videos'
WLASL_DIR = PROJECT_ROOT / 'kaggle_dataset'
OUTPUT_CSV = WLASL_DIR / 'wlasl_universal_metadata_rebuilt.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('WLASL_JSON exists:', WLASL_JSON.exists())
print('VIDEOS_DIR exists:', VIDEOS_DIR.exists())
print('WLASL_DIR:', WLASL_DIR)
print('OUTPUT_CSV:', OUTPUT_CSV)

PROJECT_ROOT: c:\Users\kacpe\source\repos\szum
WLASL_JSON exists: True
VIDEOS_DIR exists: True
WLASL_DIR: c:\Users\kacpe\source\repos\szum\kaggle_dataset
OUTPUT_CSV: c:\Users\kacpe\source\repos\szum\kaggle_dataset\wlasl_universal_metadata_rebuilt.csv


In [11]:
def probe_video(video_path: Path):
    if not video_path.exists():
        return None, None, None, None

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None, None, None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if total_frames <= 0:
        total_frames = None
    if fps <= 0:
        fps = None
    if width <= 0:
        width = None
    if height <= 0:
        height = None

    return total_frames, fps, width, height


with WLASL_JSON.open('r', encoding='utf-8') as f:
    data = json.load(f)

total = sum(len(item.get('instances', [])) for item in data)
PRINT_EVERY = 200
processed = 0

rows = []
for item in data:
    label = item.get('gloss')

    for inst in item.get('instances', []):
        processed += 1
        if processed == 1 or processed % PRINT_EVERY == 0 or processed == total:
            print(f'Processed {processed}/{total}')

        video_id = str(inst.get('video_id', '')).strip()
        video_path = VIDEOS_DIR / f'{video_id}.mp4'

        has_video = video_path.exists()
        total_frames, video_fps, video_width, video_height = probe_video(video_path)

        start_frame = inst.get('frame_start')
        raw_end = inst.get('frame_end')

        try:
            start_frame = int(start_frame)
        except (TypeError, ValueError):
            start_frame = None

        try:
            raw_end = int(raw_end)
        except (TypeError, ValueError):
            raw_end = None

        if raw_end == -1:
            end_frame = total_frames
        else:
            end_frame = raw_end

        if start_frame is not None and end_frame is not None and end_frame >= start_frame:
            length_frames = end_frame - start_frame + 1
        else:
            length_frames = None

        fps = video_fps if video_fps is not None else inst.get('fps')
        if fps is not None and fps > 0 and length_frames is not None:
            duration_sec = round(length_frames / fps, 3)
        else:
            duration_sec = None

        # Relative path from project root (szum/)
        try:
            rel_path = str(video_path.relative_to(PROJECT_ROOT)).replace('\\', '/')
        except ValueError:
            rel_path = str(video_path).replace('\\', '/')

        rows.append({
            'label': label,
            'source': inst.get('source'),
            'video_path': rel_path,
            'start_frame': start_frame,
            'end_frame': end_frame,
            'length_frames': length_frames,
            'duration_sec': duration_sec,
            'fps': fps,
            'signer_id': inst.get('signer_id'),
            'has_video': has_video,
            'video_width': video_width,
            'video_height': video_height,
        })

print('Total rows:', len(rows))

Processed 1/21083
Processed 200/21083
Processed 400/21083
Processed 600/21083
Processed 800/21083
Processed 1000/21083
Processed 1200/21083
Processed 1400/21083
Processed 1600/21083
Processed 1800/21083
Processed 2000/21083
Processed 2200/21083
Processed 2400/21083
Processed 2600/21083
Processed 2800/21083
Processed 3000/21083
Processed 3200/21083
Processed 3400/21083
Processed 3600/21083
Processed 3800/21083
Processed 4000/21083
Processed 4200/21083
Processed 4400/21083
Processed 4600/21083
Processed 4800/21083
Processed 5000/21083
Processed 5200/21083
Processed 5400/21083
Processed 5600/21083
Processed 5800/21083
Processed 6000/21083
Processed 6200/21083
Processed 6400/21083
Processed 6600/21083
Processed 6800/21083
Processed 7000/21083
Processed 7200/21083
Processed 7400/21083
Processed 7600/21083
Processed 7800/21083
Processed 8000/21083
Processed 8200/21083
Processed 8400/21083
Processed 8600/21083
Processed 8800/21083
Processed 9000/21083
Processed 9200/21083
Processed 9400/21083

In [12]:
columns = [
    'label',
    'source',
    'video_path',
    'start_frame',
    'end_frame',
    'length_frames',
    'duration_sec',
    'fps',
    'signer_id',
    'has_video',
    'video_width',
    'video_height',
]

df_all = pd.DataFrame(rows, columns=columns)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df_all.to_csv(OUTPUT_CSV, index=False)

print(f'Saved universal CSV: {OUTPUT_CSV} | rows = {len(df_all)}')

Saved universal CSV: c:\Users\kacpe\source\repos\szum\kaggle_dataset\wlasl_universal_metadata_rebuilt.csv | rows = 21083


In [13]:
print(rows[0])
print(rows[1])
print(rows[2])


{'label': 'book', 'source': 'aslbrick', 'video_path': 'kaggle_dataset/videos/69241.mp4', 'start_frame': 1, 'end_frame': 75, 'length_frames': 75, 'duration_sec': 2.502, 'fps': 29.97002997002997, 'signer_id': 118, 'has_video': True, 'video_width': 1280, 'video_height': 720}
{'label': 'book', 'source': 'aslsignbank', 'video_path': 'kaggle_dataset/videos/65225.mp4', 'start_frame': 1, 'end_frame': None, 'length_frames': None, 'duration_sec': None, 'fps': 25, 'signer_id': 90, 'has_video': False, 'video_width': None, 'video_height': None}
{'label': 'book', 'source': 'valencia-asl', 'video_path': 'kaggle_dataset/videos/68011.mp4', 'start_frame': 1, 'end_frame': None, 'length_frames': None, 'duration_sec': None, 'fps': 25, 'signer_id': 110, 'has_video': False, 'video_width': None, 'video_height': None}
